# Remedia — Modal Uyumlu Giriş Notebook'u

Bu sürüm Modal importer için küçük ve standart hücrelerle hazırlanmıştır.

1. Compute bölümünden **L4 GPU**, **4 CPU**, **8 GiB RAM** seç.
2. Mümkünse `remedia-data` Volume'unu `/mnt/remedia-data` yoluna bağla.
3. Aşağıdaki iki hücreyi sırayla çalıştır.

İlk hücre GNINA için gerekli CUDA 12 çalışma kütüphanelerini otomatik kurar.
Giriş hücresi eski cache'lenmiş Remedia kodunu temizleyerek güncel `main` sürümünü kullanır.


In [ ]:
import os
import site
import subprocess
import sys
from pathlib import Path

gpu = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
if gpu.returncode != 0:
    raise RuntimeError("GPU görünmüyor. Compute ekranından L4 seçip kernel'i yeniden başlat.")

cuda_packages = [
    "nvidia-cuda-runtime-cu12>=12.4",
    "nvidia-cublas-cu12>=12.4",
    "nvidia-cusparse-cu12>=12.3",
    "nvidia-cusolver-cu12>=11.6",
    "nvidia-curand-cu12>=10.3",
    "nvidia-nvjitlink-cu12>=12.4",
]
print("📦 GNINA için CUDA 12 çalışma kütüphaneleri doğrulanıyor...")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", *cuda_packages],
    check=True,
)

library_dirs = []
for base in [*site.getsitepackages(), site.getusersitepackages()]:
    nvidia_root = Path(base) / "nvidia"
    if nvidia_root.exists():
        library_dirs.extend(str(path) for path in nvidia_root.glob("*/lib") if path.is_dir())

existing = [p for p in os.environ.get("LD_LIBRARY_PATH", "").split(":") if p]
merged = list(dict.fromkeys([*library_dirs, *existing]))
os.environ["LD_LIBRARY_PATH"] = ":".join(merged)

cusparse = [Path(path) / "libcusparse.so.12" for path in library_dirs]
if not any(path.exists() for path in cusparse):
    raise RuntimeError("libcusparse.so.12 kurulamadı; GNINA başlatılamaz.")

print("✅ GPU ve CUDA 12 runtime hazır")


In [ ]:
import json
import shutil
import urllib.request
from pathlib import Path

for stale_repo in (Path('/mnt/remedia-data/Remedia'), Path('/root/remedia_workspace/Remedia')):
    if stale_repo.exists():
        shutil.rmtree(stale_repo)
        print(f'♻️ Eski kod cache’i temizlendi: {stale_repo}')

FORM_NOTEBOOK_URL = (
    "https://raw.githubusercontent.com/mehmetg06/Remedia/"
    "main/notebooks/remedia_modal.ipynb"
)

with urllib.request.urlopen(FORM_NOTEBOOK_URL) as response:
    source_notebook = json.load(response)

code_cells = [
    cell.get("source", "")
    for cell in source_notebook.get("cells", [])
    if cell.get("cell_type") == "code"
]
if not code_cells:
    raise RuntimeError("Remedia form kodu bulunamadı.")

def normalize_source(source):
    return "".join(source) if isinstance(source, list) else source

form_source = "\n\n".join(normalize_source(cell) for cell in code_cells)
exec(compile(form_source, FORM_NOTEBOOK_URL, "exec"), globals())
